In [1]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
from datetime import datetime, timedelta

In [ ]:
# 1. i use synthetic tactical data generation for this project to simulate anti-poaching field logs. The generated dataset will include timestamps, incident types, GPS coordinates, ranger sectors, and weather conditions. this synthetic data will help in testing and developing the wildlife intelligence analysis system without relying on sensitive real-world data.

print("Generating raw anti-poaching field logs...")
np.random.seed(42)
n_records = 800

# Generating baseline timestamps distributed across a 60-day period
start_date = datetime(2026, 7, 1)
random_days = np.random.randint(0, 60, n_records)

# Full 24-hour cycle array
hours_in_day = list(range(24))

# High night-time weight to simulate illegal incursions peaking at night/dawn
hour_probabilities = [
    0.07, 0.07, 0.07, 0.06, 0.05, 0.03,  # 00:00 - 05:00 (High activity)
    0.02, 0.02, 0.02, 0.02, 0.02, 0.02,  # 06:00 - 11:00 (Low morning)
    0.02, 0.02, 0.03, 0.03, 0.03, 0.04,  # 12:00 - 17:00 (Moderate afternoon)
    0.05, 0.06, 0.07, 0.07, 0.07, 0.07   # 18:00 - 23:00 (High evening)
]

# FIX: Normalize probabilities so they strictly sum to 1.0 in NumPy's POV
hour_probabilities = np.array(hour_probabilities)
hour_probabilities /= hour_probabilities.sum()

random_hours = np.random.choice(
    hours_in_day, 
    size=n_records, 
    p=hour_probabilities
) 
 

timestamps = [start_date + timedelta(days=int(d), hours=int(h)) for d, h in zip(random_days, random_hours)]

# Coordinates representing a realistic wildlife area (e.g., Chewore South region)
latitudes = np.random.normal(loc=-16.1, scale=0.08, size=n_records)
longitudes = np.random.normal(loc=29.9, scale=0.08, size=n_records)

incident_types = np.random.choice(
    ["Snare Detected", "Poacher Tracks", "Unauthorized Entry", "Carcass Found"], 
    size=n_records, 
    p=[0.40, 0.30, 0.20, 0.10]
)

sectors = np.random.choice(["Alpha", "Bravo", "Charlie", "Delta"], size=n_records, p=[0.20, 0.45, 0.15, 0.20])
weather = np.random.choice(["Clear", "Overcast", "Heavy Rain"], size=n_records, p=[0.60, 0.30, 0.10])

# Construct raw DataFrame with deliberate missing fields to clean
raw_df = pd.DataFrame({
    "Incident_ID": [f"INC-{1000+i}" for i in range(n_records)],
    "Timestamp": timestamps,
    "Incident_Type": incident_types,
    "Latitude": latitudes,
    "Longitude": longitudes,
    "Ranger_Sector": sectors,
    "Weather_Condition": weather
})

# Introduce some missing rows to simulate field tracking errors
missing_indices = np.random.choice(raw_df.index, size=40, replace=False)
raw_df.loc[missing_indices, 'Weather_Condition'] = np.nan

Generating raw anti-poaching field logs...


In [6]:
# 2. the data preprocessing and feature extraction phase will clean the raw dataset, handle missing values, and derive operational features such as time-of-day, day-of-week, and night patrol indicators. This step is crucial for preparing the data for subsequent analysis and visualization.

print("Processing data and extracting operational features...")
cleaned_df = raw_df.copy()

# Fix missing values
cleaned_df['Weather_Condition'] = cleaned_df['Weather_Condition'].fillna("Unknown")

# Time-series parsing to extract patrol insights
cleaned_df['Timestamp'] = pd.to_datetime(cleaned_df['Timestamp'])
cleaned_df['Hour_of_Day'] = cleaned_df['Timestamp'].dt.hour
cleaned_df['Day_of_Week'] = cleaned_df['Timestamp'].dt.day_name()

# Define the night patrol window hours (18:00 to 06:00)
night_hours = list(range(18, 24)) + list(range(0, 7))
cleaned_df['Is_Night_Patrol'] = cleaned_df['Hour_of_Day'].isin(night_hours)

Processing data and extracting operational features...


In [7]:
# 3. i will perform a tactical intelligence analysis to identify high-risk sectors, temporal patterns of illegal activity, and incident type distributions. This analysis will provide actionable insights for ranger deployment and anti-poaching strategies.

print("\n--- TACTICAL INTELLIGENCE REPORT OUTCOMES ---")

# 1. Identify highest threat sector
sector_threats = cleaned_df['Ranger_Sector'].value_counts()
print(f"Top High-Risk Sector: Sector {sector_threats.index} ({sector_threats.values} incidents resolved)")

# 2. Temporal analysis (Night vs Day activity rates)
night_incursions = cleaned_df['Is_Night_Patrol'].sum()
night_percentage = (night_incursions / len(cleaned_df)) * 100
print(f"Night-time Illegal Activity Weight: {night_percentage:.2f}% of all logs")

# 3. Incident Type breakdown
type_breakdown = cleaned_df['Incident_Type'].value_counts(normalize=True) * 100
print("\nThreat Matrix Breakdown:")
for inc_type, pct in type_breakdown.items():
    print(f" * {inc_type}: {pct:.1f}%")


--- TACTICAL INTELLIGENCE REPORT OUTCOMES ---
Top High-Risk Sector: Sector Index(['Bravo', 'Alpha', 'Delta', 'Charlie'], dtype='object', name='Ranger_Sector') ([341 178 164 117] incidents resolved)
Night-time Illegal Activity Weight: 71.00% of all logs

Threat Matrix Breakdown:
 * Snare Detected: 38.8%
 * Poacher Tracks: 31.1%
 * Unauthorized Entry: 20.5%
 * Carcass Found: 9.6%


In [9]:
# 4. here i will generate an interactive geospatial heatmap to visualize incident hotspots, enabling tactical deployment and resource allocation for anti-poaching operations.

print("\nGenerating tactical geospatial heatmap visualization...")

# Instantiate map base centered on area coordinates (Chewore South area)
base_map = folium.Map(location=[-16.1, 29.9], zoom_start=11, tiles="OpenStreetMap")

# Isolate coordinate parameters for heatmap computation
heatmap_points = cleaned_df[['Latitude', 'Longitude']].values.tolist()

# Add standard heatmap layer configuration
HeatMap(heatmap_points, radius=15, blur=10).add_to(base_map)

# Save the final file output
map_filename = "incident_hotspots_map.html"
base_map.save(map_filename)
print(f"Success! Interactive map generated and saved locally as '{map_filename}'.")


Generating tactical geospatial heatmap visualization...
Success! Interactive map generated and saved locally as 'incident_hotspots_map.html'.
